#### Done: pmf, cdf, icdf, mean, variance, skewness, mode, param_shape

In [ ]:
from typing import Optional, Tuple

import torch
from torch import Tensor
from torch.distributions import constraints
from torch.distributions.distribution import Distribution
from torch.distributions.utils import (
    lazy_property,
    logits_to_probs,
    probs_to_logits,
)
from torch.distributions import Normal
from torch.fft import fft, ifft
import math
import torch.nn.functional as F

__all__ = ["PoissonBinomialExact"]

def _clamp_by_zero(x):
    return (x.clamp(min=0) + x - x.clamp(max=0)) / 2

class PoissonBinomialExact(Distribution):
    arg_constraints = {
        "probs": constraints.unit_interval,
        "logits": constraints.real,
    }
    has_enumerate_support = True

    def __init__(
        self,
        probs: Optional[Tensor] = None,
        logits: Optional[Tensor] = None,
        validate_args: bool = True,
    ) -> None:
        if (probs is None) == (logits is None):
            raise ValueError("Either `probs` or `logits` must be specified, but not both.")
        if probs is not None:
            self.probs = probs
        else:
            self.probs = logits_to_probs(logits)
        batch_shape = self.probs.shape
        super().__init__(batch_shape, validate_args=validate_args)
    
    def _pmf(self, device='cpu'):
        probs = self.probs
        if not isinstance(probs, torch.Tensor):
            probs = torch.tensor(probs, dtype=torch.float64)
        probs = probs.to(device)
        
        if probs.dim() == 0:
            probs = probs.unsqueeze(0)
        
        n = probs.shape[-1]
        N = n + 1
        
        real_dtype = probs.dtype
        complex_dtype = torch.complex128 if real_dtype == torch.float64 else torch.complex64
        
        j = torch.arange(N, device=device, dtype=real_dtype)
        omega = torch.exp(2j * math.pi * j / N)
        
        cf = torch.ones((N,), dtype=complex_dtype, device=device)
        
        for i in range(n):
            p_i = probs[i].unsqueeze(-1)
            term = (1 - p_i) + p_i * omega
            cf = cf * term
        
        pmf = torch.fft.fft(cf, dim=-1).real / N
        pmf = pmf.clamp_(min=0.0)
        
        total = pmf.sum()
        if total > 0:
            pmf /= total
        
        return pmf.cpu()

    def _cdf(self, value=None, device='cpu'):
        pmf = self._pmf(self.probs, device=device)
        cdf_full = torch.cumsum(pmf, dim=-1)           # shape (n+1,)
    
        if value is None:
            return cdf_full
        
        # Evaluate at specific value(s)
        value = torch.as_tensor(value, dtype=torch.long, device=device)
        value = value.clamp_(min=0, max=len(pmf)-1)
        
        # Gather
        cdf_values = cdf_full[value]
        return cdf_values
    
    def _icdf(self, value=None, device='cpu'):
        pmf = self._pmf(self.probs, device=device)
        cdf_full = torch.cumsum(pmf, dim=-1)           # shape (n+1,)
    
        if value is None:
            return cdf_full
        
        value = torch.as_tensor(value, dtype=torch.float32, device=device)
        value = value.clamp_(min=0.0, max=1.0)
        
        icdf_values = torch.searchsorted(cdf_full, value)
        return icdf_values
    
    @property
    def mean(self) -> Tensor:
        return self.probs.sum(-1)
    
    @property
    def variance(self) -> Tensor:
        return (self.probs * (1 - self.probs)).sum(-1)

    @property
    def skewness(self) -> Tensor:
        p = self.probs
        term = p * (1 - p) * (1 - 2 * p)
        return term.sum(-1) / self.variance.clamp(min=1e-8).pow(1.5)
    
    @property
    def mode(self) -> Tensor:
        return self._pmf().argmax(dim=-1)
    
    @property
    def param_shape(self) -> torch.Size:
        return self._param.size()

In [ ]:
from typing import Optional, Tuple

import torch
from torch import Tensor
from torch.distributions import constraints
from torch.distributions.distribution import Distribution
from torch.distributions.utils import (
    lazy_property,
    logits_to_probs,
    probs_to_logits,
)
from torch.distributions import Normal
from torch.fft import fft, ifft
import math
import torch.nn.functional as F

__all__ = ["PoissonBinomialApprox"]

def _clamp_by_zero(x):
    return (x.clamp(min=0) + x - x.clamp(max=0)) / 2

class PoissonBinomialApprox(Distribution):
    arg_constraints = {
        "probs": constraints.unit_interval,
        "logits": constraints.real,
    }
    has_enumerate_support = True

    def __init__(
        self,
        probs: Optional[Tensor] = None,
        logits: Optional[Tensor] = None,
        validate_args: bool = True,
    ) -> None:
        if (probs is None) == (logits is None):
            raise ValueError("Either `probs` or `logits` must be specified, but not both.")
        if probs is not None:
            self.probs = probs
        else:
            self.probs = logits_to_probs(logits)
        batch_shape = self.probs.shape
        super().__init__(batch_shape, validate_args=validate_args)
    
    def _pmf(self, device='cpu'):
        probs = self.probs
        if not isinstance(probs, torch.Tensor):
            probs = torch.tensor(probs, dtype=torch.float64)
        probs = probs.to(device).flatten()
        n = probs.numel()

        if n == 0:
            return torch.tensor([1.0], device=device)

        mu = probs.sum()
        var = probs.mul(1 - probs).sum()

        if var <= 1e-12:
            pmf = torch.zeros(n + 1, device=device, dtype=torch.float64)
            pmf[round(mu.item())] = 1.0
            return pmf.clamp_(min=0)

        sigma = torch.sqrt(var)
        mu3 = probs.mul(1 - probs).mul(1 - 2 * probs).sum()
        skew = mu3 / (sigma ** 3)                
        k = torch.arange(n + 1, device=device, dtype=torch.float64)
        z = (k - mu) / sigma

        phi = torch.exp(-0.5 * z * z) / math.sqrt(2 * math.pi)          # pdf
        Phi = torch.distributions.Normal(0, 1).cdf(z)                   # cdf

        correction = (skew / 6.0) * (z * z - 1.0) * phi
        pdf_approx = phi + correction
        pmf = pdf_approx.clamp_(min=0.0)

        total = pmf.sum()
        if total > 1e-10:
            pmf.div_(total)

        return pmf.cpu()

    def _cdf(self, value=None, device='cpu'):
        pmf = self._pmf(self.probs, device=device)
        cdf_full = torch.cumsum(pmf, dim=-1)           # shape (n+1,)
    
        if value is None:
            return cdf_full
        
        # Evaluate at specific value(s)
        value = torch.as_tensor(value, dtype=torch.long, device=device)
        value = value.clamp_(min=0, max=len(pmf)-1)
        
        # Gather
        cdf_values = cdf_full[value]
        return cdf_values
    
    def _icdf(self, value=None, device='cpu'):
        pmf = self._pmf(self.probs, device=device)
        cdf_full = torch.cumsum(pmf, dim=-1)           # shape (n+1,)
    
        if value is None:
            return cdf_full
        
        value = torch.as_tensor(value, dtype=torch.float32, device=device)
        value = value.clamp_(min=0.0, max=1.0)
        
        icdf_values = torch.searchsorted(cdf_full, value)
        return icdf_values.cpu()
    
    @property
    def mean(self) -> Tensor:
        return self.probs.sum(-1)
    
    @property
    def variance(self) -> Tensor:
        return (self.probs * (1 - self.probs)).sum(-1)

    @property
    def skewness(self) -> Tensor:
        p = self.probs
        term = p * (1 - p) * (1 - 2 * p)
        return term.sum(-1) / self.variance.clamp(min=1e-8).pow(1.5)

    @property
    def mode(self) -> Tensor:
        return self._pmf().argmax(dim=-1)

    @property
    def param_shape(self) -> torch.Size:
        return self._param.size()